In [0]:
%sql
/*Create a Volume*/
CREATE VOLUME IF NOT EXISTS demo_catalog.bronze.02etlpipeline_files;
SHOW VOLUMES IN demo_catalog.bronze;

In [0]:
#set schema path and checkpoint path
schema_path = "/Volumes/demo_catalog/bronze/raw_files/_schema"
checkpoint_path = "/Volumes/demo_catalog/bronze/raw_files/_checkpoint"

In [0]:
#Read the data from csv file to dataframe
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/demo_catalog/bronze/02etlpipeline_files/customers.csv")

df.show()

In [0]:
#Apply transformations
from pyspark.sql.functions import *
df_transformed = df \
    .withColumn("name", initcap(col("name"))) \
    .withColumn("city", coalesce(col("city"), lit("Unknown"))) \
    .withColumn("age", coalesce(col("age"), lit(0))) \
    .withColumn(
        "signup_date",
        to_date(col("signup_date"), "dd-MM-yyyy")
    ) \
    .withColumn("ingestion_ts", current_timestamp())\
    .withColumn("filename", col("_metadata.file_path"))
#Validate Transformations
df_transformed.show(truncate=False)
#df_transformed.printSchema()   

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, to_date, current_timestamp

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("signup_date", StringType(), True),
])
df = (
    spark.readStream
         .format("csv")
         .schema(schema)
         .option("header", "true")
         .option("path", "/Volumes/demo_catalog/bronze/02etlpipeline_files/")
         .load()
         .withColumn("customer_id", col("customer_id").cast("int"))
         .withColumn("name", col("name").cast("string"))
         .withColumn("city", col("city").cast("string"))
         .withColumn("age", col("age").cast("int"))
         .withColumn("signup_date", to_date(col("signup_date"), "dd-MM-yyyy"))
         .withColumn("source_file", col("_metadata.file_path"))
         .withColumn("ingestion_ts", current_timestamp())
)
query = (
    df.writeStream
      .format("memory")
      .queryName("customers")
      .option("checkpointLocation", checkpoint_path)
      .trigger(availableNow=True)
      .start()
)
query.awaitTermination()
spark.sql("SELECT * FROM customers").show(truncate=False)

In [0]:
#display(dbutils.fs.ls("/Volumes/demo_catalog/bronze/02etlpipeline_files/"))
dbutils.fs.rm(checkpoint_path, True)
print(query.status)
print(query.lastProgress)